# ACX3300 Project 1: Forecasting Earnings

**Team Members:**
- Yizhou Hu (36076406)
- Phong Phan (33867003)
- Jiaye Yuan (Student ID 3)

## Table of Contents

wip-(2132)

## Part 1: Data Preparation

The purpose of the project is to forecast future firm earnings using accounting data. We examine whether past earnings information and various financial variables can be used to effectively predict future earnings performance. This is important as the forecast assists investors, analysts and stakeholders in evaluating firm performance, profitability and future value.

This project applies both time series forecasting models and accounting-based forecasting models. The time series models include random walk model and moving average models. The random walk model assumes that future earnings per share can be predicted using current earnings per share (EPS). Moving average models are also used to smooth short-term fluctuations in earnings by averaging EPS over the past 2, 3 and 5 years.

The project includes an OLS regression model to forecast future profitability. The model examines whether current profitability and firm characteristics help explain future earnings performance.

The conceptual regression model is:

$$
ROA_{t+1} = \beta_0 + \beta_1 ROA_t + \beta_2 PM_t + \beta_3 Loss_t + \beta_4 \log AT_t + \beta_5 ATO_t + \varepsilon
$$

More specifically, this regression predicts next year's ROA ($ROA_{t+1}$) using current variables observed in year $t$.

### Load & Explore Raw Data

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

df = pd.read_csv('accounting_data.csv')
n1 = len(df)
df.head()

### Data Cleaning

Before modeling, we remove observations that lack essential accounting
variables, sort the panel into firm-year order, and filter out economically
implausible records.

The core variables — `gvkey` (firm identifier), `fyear` (fiscal year),
`at` (total assets), `ceq` (common equity), `ni` (net income),
`sale` (sales revenue), `csho` (shares outstanding), `dlc` (debt in
current liabilities), and `dltt` (long-term debt) — are the raw
inputs from which all downstream ratios (ROA, profit margin, EPS, log
assets, asset turnover, leverage) are constructed. If any of these fields is missing, the
observation cannot contribute to the analysis, so `dropna(subset=...)`
removes those rows directly. `n2` tracks the post-cleaning sample size
for the `selection_summary` audit trail.

We then sort by `['gvkey', 'fyear']` to establish the chronological
panel structure. Correct ordering is a prerequisite for every
time-series operation that follows — `shift()`, `groupby().rolling()`,
and multi-period forecast horizons all depend on knowing that row `i`
immediately precedes row `i+1` for the same firm.

Finally, we impose basic economic filters: total assets, common equity,
and sales revenue must all be positive, and shares outstanding cannot be
zero. These constraints eliminate firm-years that are either bankrupt,
pre-revenue, or affected by data-entry errors — observations that do not
reflect normal operating conditions and would distort the statistical
estimates. `n3` records the sample size entering the winsorization step
that follows.

In [ ]:
essential_cols = ['gvkey', 'fyear', 'at', 'ceq', 'ni', 'sale', 'csho', 'dlc', 'dltt']
df.dropna(subset=essential_cols, inplace=True)

df.sort_values(by=['gvkey', 'fyear'], inplace=True)
n2 = len(df)

df = df[(df['at'] > 0) & (df['ceq'] > 0) & (df['sale'] > 0) & (df['csho'] != 0)]
n3 = len(df)

### Winsorization

Extreme outliers in the distribution tails of financial variables can
disproportionately influence regression estimates and correlation coefficients,
even after removing economically impossible values in the prior cleaning step.
To address this, we apply winsorization — capping observations below the 1st
percentile and above the 99th percentile rather than deleting them. The 1%/99%
threshold is standard in accounting research, retaining approximately 98% of
the original data variation while limiting outlier influence.

The `wins(df, vars)` function implements this logic. For each variable in the
`vars` list, it computes the 1st and 99th percentiles within each fiscal year
using `groupby('fyear')[var].transform(clip_series)` and applies `clip()` to
restrict values to this range. Year-by-year winsorization is essential because
financial statement magnitudes shift with the economic cycle — inflation and
business cycle fluctuations affect the absolute levels of assets, income, and
sales across periods. Pooling all years would assume a stationary distribution
and risk inappropriate trimming during booms or downturns.

The seven variables — `at`, `ni`, `sale`, `ceq`, `csho`, `dlc`, `dltt` —
are the raw components from which all downstream ratios (ROA, profit
margin, EPS, log assets, asset turnover, leverage) are constructed.
Winsorizing at the source ensures data consistency
throughout the entire analysis pipeline. After transformation, `.dropna()`
removes any newly created missing values, and `n4` tracks the sample size for
the `selection_summary` audit trail.

Finally, `winsor_check` displays the post-winsorization minima and maxima of
total assets for the first ten fiscal years, providing a diagnostic
verification that the clipping was applied correctly.

In [ ]:
def wins(df, vars):
    def clip_series(s):
        lower = s.quantile(0.01)
        upper = s.quantile(0.99)
        return s.clip(lower=lower, upper=upper)
    for var in vars:
        df[var] = df.groupby('fyear')[var].transform(clip_series)
    return df

cols_to_winsorize = ['at', 'ni', 'sale', 'ceq', 'csho', 'dlc', 'dltt']
df = wins(df, cols_to_winsorize).dropna()
n4 = len(df)

print("Yearly Winsorization complete.")

winsor_check = df.groupby('fyear')[['at']].agg(['min', 'max']).head(10)
display(winsor_check)

### Calculate Variables

The dependent variable in the model is future return on assets ($ROA_{t+1}$), which measures future profitability. It is calculated as next year's net income divided by current total assets:

$$
ROA_{t+1} = \frac{\text{Net Income}_{t+1}}{\text{Total Assets}_t}
$$

This variable is used because ROA measures how efficiently a firm generates earnings from its asset base and is commonly used as a measure of firm performance.

Current return on assets ($ROA_t$) is included as an explanatory variable because profitability is often persistent over time, meaning firms with strong current profitability may continue to perform well in future periods. It is calculated as:

$$
ROA_t = \frac{\text{Net Income}_t}{\text{Total Assets}_t}
$$

Profit margin ($PM_t$) is included because it measures operating profitability and shows how much profit a firm generates from each dollar of sales revenue. Firms with higher profit margins may have stronger and more sustainable future earnings. It is calculated as:

$$
PM_t = \frac{\text{Net Income}_t}{\text{Sales Revenue}_t}
$$

Firm size ($\log AT_t$) is included because larger firms are generally more stable and may have more predictable earnings patterns. Firm size is measured using the natural logarithm of total assets to reduce skewness caused by extremely large firms:

$$
\log AT_t = \ln(\text{Total Assets}_t)
$$

Furthermore, it aligns the model with the business reality of diminishing marginal effects. Without a logarithmic transformation, a linear model assumes that an absolute asset increase of $10 million has the exact same impact on a $10 million firm as it does on a $100 billion corporate giant. Taking the natural logarithm ensures the model evaluates proportional, rather than absolute, growth, correcting this violation of business common sense.

The loss indicator ($Loss_t$) is included to capture whether a firm reports negative earnings. Firms reporting losses may experience lower earnings persistence and greater uncertainty surrounding future profitability. The variable is constructed as:

$$
Loss_t = \begin{cases} 1 & \text{if Net Income}_t < 0 \\ 0 & \text{otherwise} \end{cases}
$$

Asset turnover ($ATO_t$) is included because it measures how efficiently a firm utilizes its assets to generate sales revenue. Firms with higher asset turnover generate more revenue per dollar of assets, reflecting stronger operational efficiency. This variable captures a dimension of firm performance — the ability to convert asset investment into revenue — that is distinct from profitability ratios such as ROA and profit margin. It is calculated as:

$$
ATO_t = \frac{\text{Sales Revenue}_t}{\text{Total Assets}_t}
$$

Leverage ($LEV_t$) is included because a firm's capital structure — the extent to which it relies on debt financing — may influence both its current performance and future earnings prospects. Higher leverage imposes fixed interest obligations that can amplify profitability fluctuations and increase financial risk. It is measured as total debt (the sum of debt in current liabilities and long-term debt) divided by total assets:

$$
LEV_t = \frac{\text{Debt in Current Liabilities}_t + \text{Long-Term Debt}_t}{\text{Total Assets}_t}
$$

The following code constructs the dependent and independent variables for the OLS regression model and prepares earnings per share (EPS) variables for the time series forecasting models.

Future net income ($NI_{t+1}$) is obtained by shifting the net income series one year forward within each firm using `groupby('gvkey')['ni'].shift(-1)`. The `groupby` operation ensures the shift is applied separately for each firm, preventing cross-firm data leakage. Observations without a next-year net income value — the final year for each firm — are dropped. The OLS variables ($ROA_{t+1}$, $ROA_t$, $PM_t$, $\log AT_t$, $Loss_t$, $ATO_t$, $LEV_t$) are then computed from $NI_{t+1}$ and the winsorized raw variables according to the formulas introduced in the Data Preparation section above.

For the time series forecasting models (random walk and moving average), current and future EPS values are constructed:

$$EPS_t = \frac{NI_t}{\text{CSHO}_t}$$

Future EPS values ($EPS_{t+1}$, $EPS_{t+2}$, $EPS_{t+3}$) are obtained by shifting the EPS series forward by one, two, and three periods within each firm:

$$EPS_{t+k} = \frac{NI_{t+k}}{\text{CSHO}_{t+k}} \quad \text{for } k = 1, 2, 3$$

The `groupby('gvkey')['eps'].shift(-k)` operation again maintains firm-level boundaries so that EPS values are never shifted across different companies.

After the ratios are computed, any infinite values — caused by division by zero in intermediate calculations — are replaced with `NaN` and subsequently dropped. Observations with missing values in any key variable (`roa_t1`, `pm_t`, `eps`, `eps_t1`, `eps_t2`, `eps_t3`, `ato_t`, `lev_t`) are removed to ensure a complete and consistent dataset for all subsequent analyses.

Finally, a `selection_summary` DataFrame is constructed to track the number of observations remaining after each filtering step, from the raw data through winsorization and variable construction. This provides a transparent audit trail of how the sample evolves from 285,491 initial observations to the final analytical sample.


In [ ]:
df['ni1'] = df.groupby('gvkey')['ni'].shift(-1)
df.dropna(subset=['ni1'], inplace=True)

df['roa_t1'] = df['ni1'] / df['at']
df['pm_t']  = df['ni'] / df['sale']
df['roa_t'] = df['ni'] / df['at']
df['log_at'] = np.log(df['at'])
df['loss_t'] = np.where(df['ni']<0, 1, 0)
df['ato_t'] = df['sale'] / df['at']
df['lev_t'] = (df['dlc'] + df['dltt']) / df['at']

df['eps'] = df['ni'] / df['csho']
df['eps_t1'] = df.groupby('gvkey')['eps'].shift(-1)
df['eps_t2'] = df.groupby('gvkey')['eps'].shift(-2)
df['eps_t3'] = df.groupby('gvkey')['eps'].shift(-3)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(subset=['roa_t1', 'pm_t', 'eps', 'eps_t1', 'eps_t2', 'eps_t3', 'ato_t', 'lev_t'], inplace=True)
n5 = len(df)

selection_summary = pd.DataFrame({
    'Filtering Step': [
        '1. Raw Data from accounting_data.csv',
        '2. Drop missing essential fields',
        '3. Economic Logic (at, ceq, sale > 0; csho != 0)',
        '4. Winsorize raw variables at 1%/99% by year',
        '5. Calculate ratios & EPS, remove Inf/NaN'
    ],
    'Observations Remaining': [n1, n2, n3, n4, n5],
    'Observations Dropped': [0, n1 - n2, n2 - n3, n3 - n4, n4 - n5]
})

selection_summary['Observations Remaining'] = selection_summary['Observations Remaining'].apply(lambda x: f'{x:,}')
selection_summary['Observations Dropped'] = selection_summary['Observations Dropped'].apply(lambda x: f'{x:,}')

display(selection_summary)

Step 1 to 2: Drop missing essential fields (−100,018)

This is the largest single-step drop, losing 35.0% of the original sample. These observations are missing at least one of the nine essential fields: gvkey, fyear, at, ceq, ni, sale, csho, dlc, or dltt. In Compustat, incomplete reporting is common — many firms, especially smaller ones, don't report every balance sheet and income statement item. Losing over 100,000 observations here tells us the raw data has a big completeness problem. Still, this step is non-negotiable: without these fields, none of the downstream ratios (ROA, PM, EPS, ATO, LEV) can be built.

Step 2 to 3: Economic logic filter (−38,754)

This step drops firms with at ≤ 0, ceq ≤ 0, sale ≤ 0, or csho = 0. These are firms in severe distress (negative equity, no revenue), data errors, or entities that aren't normal operating businesses. The 38,754 dropped (13.6% of original) are records that don't carry useful information for earnings prediction. If we kept them, these extreme values would mess up the regression coefficients and correlations later.

Step 3 to 4: Dropna after winsorization (−38,222)

After winsorizing the seven raw variables (at, ni, sale, ceq, csho, dlc, dltt) at 1%/99% by fiscal year, dropna() removes another 38,222 rows (13.4%). This can happen when a fiscal year has too few observations for percentile calculation, or when values were already NaN before winsorizing. The number is close to Step 3's drop (38,754), so this step deals with a similar-sized batch of problem observations, just from a different cause.

Step 4 to 5: Dropna after constructing ratios and EPS (−46,986)

This is the second biggest drop (16.5%). The dropna() across eight variables — roa_t1, pm_t, eps, eps_t1, eps_t2, eps_t3, ato_t, lev_t — combines several sources of data loss.

First, the time-series truncation effect. The groupby('gvkey').shift(-2) and .shift(-3) calls mean each firm needs at least two and three future years of data. Firms near the end of the sample window can't produce eps_t2 or eps_t3, so their rows get dropped entirely. In practice, each firm needs at least four consecutive years (current + three ahead) to survive this filter.

Second, missing next-period net income. If ni1 (from shift(-1)) is unavailable, roa_t1 can't be computed, and the row is gone.

Third, extreme values from ratio division. Things like ni/sale or ni/at can produce Inf when denominators get close to zero, and those get cleared by replace([inf, -inf], np.nan).

Fourth, the compounding effect. Since dropna() requires all eight variables to be non-missing at once, a single missing value in any one variable takes down the whole row. This stacking effect is why Step 5's reduction (46,986) is second only to Step 2 (100,018).


### Winsorization on Ratio

In the previous data preparation step, although we winsorized the raw
financial variables — at, ni, sale, ceq, csho, dlc, and dltt — at the 1st and 99th
percentiles by fiscal year, we still observe extreme values in the ratio
variables computed from these winsorized components (roa_t1, pm_t,
roa_t, ato_t, and lev_t).

Specifically, describe() reveals that roa_t1 reaches a maximum of 25.09,
pm_t reaches 414.17, and roa_t reaches 24.59. By comparison, the 75th
percentiles stand at only 0.079, 0.112, and 0.071 respectively — the
maximum values exceed the upper end of the reasonable range by several
orders of magnitude. These values carry no economic meaning (a pm_t of
414 implies $414 of net income per dollar of sales revenue, which is
impossible in reality). If left untreated, they would severely distort
the subsequent Pearson correlation analysis and OLS regression estimates.

We believe these extreme ratio values are not driven by outliers in the
underlying raw variables, but rather arise from the boundary combination
effect. Winsorization guarantees that each individual variable falls within
the [1%, 99%] interval, but it does not prevent two boundary values from
producing an extreme result when combined through division. For instance,
if a firm's net income (ni) happens to lie near the 1st-percentile boundary
while its sales revenue (sale) lies near the 99th-percentile boundary, the
ratio pm_t = ni / sale can produce an artificially extreme value. The root
cause is that the numerator and denominator are trimmed independently and
asynchronously — the clipping boundaries are not coordinated across variables.

To address this issue, we apply a second round of winsorization directly to
the five ratio variables themselves.

First, a boolean switch ENABLE_RATIO_WINSOR controls whether this step is
executed, which facilitates debugging and before/after comparison. When
enabled, we call describe() on the five variables to display their
distribution prior to winsorization, making the extreme values visible.

Next, we iterate over the ratio_vars list containing roa_t1, pm_t, roa_t,
ato_t, and lev_t. For each variable, we apply groupby('fyear')[var].transform to
compute the 1st and 99th percentiles independently within each fiscal year,
then use clip to trim observations beyond these bounds. We compute
percentiles by fiscal year because financial ratio distributions shift with
the economic cycle — pooling across years for percentile calculation could
lead to inappropriate trimming in certain periods.

Finally, we call describe() again on the winsorized variables to verify that
the extreme values have been successfully removed.

In [ ]:
ENABLE_RATIO_WINSOR = True
if ENABLE_RATIO_WINSOR:
    print("Before:")
    from IPython.display import display
    display(df[['roa_t1','pm_t','roa_t','ato_t','lev_t']].describe())

    ratio_vars = ['roa_t1', 'pm_t', 'roa_t', 'ato_t', 'lev_t']
    for var in ratio_vars:
        df[var] = df.groupby('fyear')[var].transform(
            lambda s: s.clip(s.quantile(0.01), s.quantile(0.99))
        )

    print("After:")
    display(df[['roa_t1','pm_t','roa_t','ato_t','lev_t']].describe())

wip-description-of-the-result (3462)

result: roa_t1	pm_t	roa_t	ato_t	lev_t 的最大值从 25.090827	414.170000	24.586291	20.778330	1.004125 下降到了 0.669820	0.773579	0.393281	4.459393	0.743265


## Part 2: Descriptive Statistics and Correlation Coefficients

### Descriptive Statistics

In [ ]:
desc = df[['roa_t1', 'pm_t', 'eps', 'eps_t1', 'eps_t2', 'eps_t3', 'ato_t', 'lev_t']].describe()
display(desc)

### Time Trend of Key Financial Ratios

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(22, 14))

sns.lineplot(data=df, x='fyear', y='pm_t', marker='o', ax=axes[0, 0])
axes[0, 0].set_title('Profit Margin (pm_t)')

sns.lineplot(data=df, x='fyear', y='roa_t', marker='o', ax=axes[0, 1])
axes[0, 1].set_title('Return on Assets (roa_t)')

sns.lineplot(data=df, x='fyear', y='ato_t', marker='o', ax=axes[1, 0])
axes[1, 0].set_title('Asset Turnover (ato_t)')

sns.lineplot(data=df, x='fyear', y='lev_t', marker='o', ax=axes[1, 1])
axes[1, 1].set_title('Leverage (lev_t)')

plt.tight_layout()
plt.show()

wip-description-of-the-result (2325)

result: 2001年 pm 和 roa 特别低；2008 年有pm 和 roa 的下降和 ato 和 lev 的上升

### Correlation Matrices

Before estimating the OLS regression model, we first examine the pairwise relationships among the key variables using correlation matrices. This step serves two analytical purposes.

First, it provides a preliminary assessment of whether each independent variable — $ROA_t$, $PM_t$, $\log AT_t$, $Loss_t$, and $ATO_t$ — exhibits a meaningful association with the dependent variable $ROA_{t+1}$. A regressor that shows negligible correlation with future profitability is unlikely to contribute explanatory power in the regression; identifying such variables early avoids including noise predictors in the model.

Second, it functions as a diagnostic for multicollinearity among the independent variables. If two regressors are highly correlated — for instance, $ROA_t$ and $PM_t$ both incorporate net income in their numerators — the OLS estimator becomes unstable and the coefficient standard errors inflate, making it impossible to reliably separate each variable's individual effect. Detecting such redundancy before estimation guides variable selection decisions.

We compute both Pearson and Spearman correlation coefficients because they capture fundamentally different types of association. The Pearson coefficient measures the strength of a linear relationship — it assumes the variables move together in a straight-line fashion. The Spearman (rank) coefficient measures the strength of a monotonic relationship — it requires only that when one variable increases, the other consistently increases or decreases, without demanding linearity. This distinction is particularly relevant for the loss indicator `loss_t`, a binary variable whose relationship with other accounting metrics cannot be linear by construction. Presenting both matrices side by side allows us to judge whether linear modeling (OLS) is appropriate or whether non-linear transformations of certain variables might be warranted.

The variable set includes all seven OLS regression variables — `roa_t1`, `roa_t`, `pm_t`, `log_at`, `loss_t`, `ato_t`, and `lev_t` — plus `eps` (earnings per share). EPS is included because it serves as the central variable in the time-series forecasting models (random walk and moving average) discussed in the subsequent Conceptual Models section. Understanding EPS's correlation with the other accounting variables provides context for comparing the relative strengths of the two modeling frameworks.

The eight variables are selected into `corr_vars` and the Pearson correlation matrix is computed via `df[corr_vars].corr()`. For the Spearman case, `method='spearman'` is passed to the same function.

The heatmap visualization employs `np.triu` (upper triangular mask) to blank out the mirror-image entries above the diagonal. Correlation matrices are symmetric by construction — the correlation between $X$ and $Y$ equals that between $Y$ and $X$ — so displaying both halves creates unnecessary visual clutter. The `annot=True` parameter prints exact numeric values on each cell (`fmt='.2f'` restricts display to two decimal places), making precise coefficients readable without visually approximating the color gradient. The `cmap='Reds'` colormap encodes correlation strength through color intensity, with darker shades indicating stronger associations.

For statistical inference, we compute the p-value associated with each pairwise correlation. The p-value matrix is obtained by passing a lambda function to `df.corr()` that calls `scipy.stats.pearsonr(x, y)[1]` (or `spearmanr(x, y)[1]` for the Spearman case) to extract only the p-value from each pairwise test. A custom function `significance_stars(p)` appends conventional significance notation — `*` for $p \leq 0.001$, `` for $p \leq 0.01$, and `*` for $p \leq 0.05$ — to each coefficient. The diagonal elements are set to 1 with `np.fill_diagonal(pval.values, 1)` because a variable's correlation with itself is tautologically 1 and has an undefined p-value. These annotated matrices (`rho_stars` and `corr2_stars`) are displayed below their respective heatmaps, enabling identification of which correlations are statistically robust versus those potentially attributable to sampling noise.

The correlation results directly inform the variable selection for the OLS regression in the subsequent Conceptual Models section. Variables showing weak or statistically insignificant correlation with $ROA_{t+1}$ may be excluded from the regression specification, while highly correlated regressor pairs warrant consideration of model simplification.

#### Pearson Correlation

In [ ]:
from scipy.stats import pearsonr

corr_vars = ['roa_t1', 'roa_t', 'pm_t', 'log_at', 'loss_t', 'eps', 'ato_t', 'lev_t']

rho = df[corr_vars].corr()

mask = np.triu(np.ones_like(rho, dtype=bool))
plt.figure(figsize=(10, 8))
sns.heatmap(rho, mask=mask, annot=True, cmap='Reds', fmt='.2f')
plt.title('Pearson Correlation Matrix')
plt.show()

pval = df[corr_vars].corr(
    method=lambda x, y: pearsonr(x, y)[1]
)

np.fill_diagonal(pval.values, 1)

def significance_stars(p):
    if p <= 0.001:
        return '***'
    elif p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    return ''

stars = pval.map(significance_stars)

rho_stars = rho.round(2).astype(str) + stars
rho_stars = rho_stars.where(~mask, '')
display(rho_stars)

#### Spearman Correlation

In [ ]:
from scipy.stats import spearmanr

corr2 = df[corr_vars].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr2, mask=mask, cmap='Reds', annot=True, fmt='.2f')
plt.title('Spearman Correlation Matrix')
plt.show()

pval_spearman = df[corr_vars].corr(
    method=lambda x, y: spearmanr(x, y)[1]
)
np.fill_diagonal(pval_spearman.values, 1)

stars_spearman = pval_spearman.map(significance_stars)
corr2_stars = corr2.round(2).astype(str) + stars_spearman
corr2_stars = corr2_stars.where(~mask, '')
display(corr2_stars)

In the Pearson and Spearman correlation matrices, lev_t shows a noticeably weak connection to roa_t1 compared to the other variables. Pearson gives 0.05 and Spearman gives -0.01. For reference, roa_t sits at 0.62/0.66, pm_t at 0.48/0.48, loss_t at -0.46/-0.49, and ato_t at 0.18/0.29. With correlations that low, lev_t isn't likely to add much explanatory power to the OLS model.

There's also a mild overlap between lev_t and log_at — Pearson 0.26, Spearman 0.34 — which makes sense given that larger firms generally hold more debt. While not a serious collinearity problem by itself, it does mean lev_t and log_at share some information, so keeping both offers less incremental value.

The sign difference between Pearson and Spearman (slightly positive vs. slightly negative) also suggests the relationship between leverage and future profitability isn't particularly stable, or at least not something a simple linear coefficient can cleanly capture.

Given the weak correlation with the dependent variable and the overlap with log_at, we leave lev_t out of the regression and keep roa_t, pm_t, loss_t, log_at, and ato_t.

## Part 3: Simple Models

### Multi-period RW & Moving Average

In the code below, we set out to compare two models—Random Walk (RW) and Moving Average (MA)—and see which does a better job forecasting earnings per share (EPS).

First, we need to construct the absolute forecast error variables for each model. For the random walk models, we use the abs() function to calculate the absolute difference between actual EPS one year ahead (eps_t1), two years ahead (eps_t2), and three years ahead (eps_t3) versus the current period's EPS. This gives us our variables: error_rw_1, error_rw_2, and error_rw_3.

Second, we calculate the forecast errors for the moving average models. Since we need rolling company-level calculations, we start by grouping the EPS data by firm identifier (gvkey) using groupby('gvkey'). Then we define a custom function, calc_moving_avg, which uses pandas' rolling and mean methods to compute the 2-year, 3-year, and 5-year moving averages. We then use those moving averages (ma_2, ma_3, ma_5) as the forecasts, subtract the actual EPS one year ahead (eps_t1), and take the absolute value. That produces our error variables: error_ma_2, error_ma_3, and error_ma_5.

Finally, to make sense of the output and figure out which model is more accurate, we extract the median of all the absolute errors using the median() method. In financial data, the median usually tells a more meaningful story than the mean because it isn't thrown off by extreme outliers. Once we have the medians, we plot them as a bar chart, tapping into matplotlib and seaborn's barplot functionality. This gives us a clean, visual way to compare the forecast errors across models at a glance.

In [ ]:
df['error_rw_1'] = abs(df['eps_t1'] - df['eps'])
df['error_rw_2'] = abs(df['eps_t2'] - df['eps'])
df['error_rw_3'] = abs(df['eps_t3'] - df['eps'])

grouped_eps = df.groupby('gvkey')['eps']

def calc_moving_avg(eps_series, window_size):
    return eps_series.rolling(window=window_size, min_periods=window_size).mean()

df['ma_2'] = grouped_eps.transform(calc_moving_avg, window_size=2)
df['ma_3'] = grouped_eps.transform(calc_moving_avg, window_size=3)
df['ma_5'] = grouped_eps.transform(calc_moving_avg, window_size=5)

df['error_ma_2'] = abs(df['eps_t1'] - df['ma_2'])
df['error_ma_3'] = abs(df['eps_t1'] - df['ma_3'])
df['error_ma_5'] = abs(df['eps_t1'] - df['ma_5'])

error_cols = ['error_rw_1', 'error_rw_2', 'error_rw_3', 'error_ma_2', 'error_ma_3', 'error_ma_5']
df_errors = df.dropna(subset=error_cols)

error_medians = df_errors[error_cols].median()

display(error_medians.to_frame("Median Absolute Error"))

plt.figure(figsize=(10, 6))
ax = sns.barplot(x=error_medians.index, y=error_medians.values, palette='viridis', hue=error_medians.index, legend=False)
for c in ax.containers:
    ax.bar_label(c, fmt="%.4f")
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
plt.title('Median Absolute Forecast Error by Model')
plt.xlabel('Model')
plt.ylabel('Median Absolute Forecast Error')
plt.show()

Here are three key takeaways:

1. RW_3 has larger errors than RW_2 and RW_1

The random walk model that projects one year ahead (RW_1) gives us a much smaller average forecast error than the versions that look two or three years out (RW_2 and RW_3). Why does the accuracy drop as we push the forecast further into the future? One reason is that uncertainty in the business environment just keeps piling up over time. In the short run, a company's accounting data and its scale of operations are fairly stable. But once you stretch the horizon, more uncontrollable shocks come into play—think macro swings or shifting competitive dynamics. What this tells us is that the further you get from today, the less historical earnings can actually tell you.

2. MA models have larger errors than RW models

The moving average (MA) models actually seem to produce larger average forecast errors than the random walk (RW) models. How can a model that works with more information be less accurate than a dead-simple RW model? One possibility is that in efficient markets, a firm's current earnings already reflect the most up-to-date operating reality. Pulling out older earnings and averaging them into today's picture doesn't add explanatory power—instead, it introduces stale noise. That noise pulls the forecast away from the firm's true earnings benchmark.

3. MA_5 has larger errors than MA_3 and MA_2

Zooming in on the moving average windows, we can also see that MA_5 generates noticeably larger forecast errors than MA_3 and MA_2. In other words, stretching the calculation window out to five years didn't make the model more accurate—it just added more error. Really old accounting data (think profits from five years ago) has almost no relevance for predicting a company's future financial performance. The core insight here is that financial information decays fast. The most recent set of financials already captures all the relevant history and the current reality.

### Sub-sample Analysis by Firm Size

To dig deeper into whether firm size affects each model's forecasting accuracy, we split the sample into two groups—Large and Small—and compared their forecast error performance across different models.

Here's how we did the grouping. First, we computed the median of total assets (`at`) for each fiscal year using `groupby('fyear').transform('median')`. We did it year by year because total asset levels tend to drift upward over time with economic growth and inflation—comparing across years directly could distort the grouping. Then we used `np.where` to label each observation: if total assets were equal to or above that year's median asset level, the firm was classified as Large; otherwise, it went into the Small group.

For the error summary, we started by using `dropna` to make sure none of the error columns or the `size_group` variable contained missing values. Then we grouped by `size_group` and applied `median()` to compute the median absolute forecast error for each model separately for the Large and Small groups. We went with the median over the mean here because financial data can still harbor some extreme values. The median is a robust statistic—it doesn't get pulled around by outliers—so it gives a truer picture of the typical forecast error.

Finally, we reshaped the summary results into long format with `melt()` and plotted a grouped bar chart using `sns.barplot`. The x-axis shows the model names, the y-axis shows the median absolute error, and the hue separates the Large and Small groups. This gives us a clean visual comparison of how forecast errors differ between firms of different sizes.

In [ ]:
median_at_per_year = df.groupby('fyear')['at'].transform('median')
df['size_group'] = np.where(df['at'] >= median_at_per_year, 'Large', 'Small')

all_error_cols = ['error_rw_1', 'error_rw_2', 'error_rw_3',
                  'error_ma_2', 'error_ma_3', 'error_ma_5']
sub_df = df.dropna(subset=all_error_cols + ['size_group'])

median_errors = sub_df.groupby('size_group')[all_error_cols].median()

display(median_errors)

median_errors_reset = median_errors.reset_index().melt(
    id_vars='size_group', var_name='Model', value_name='Absolute Error'
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=median_errors_reset, x='Model', y='Absolute Error',
            hue='size_group', palette='Set2')
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f")
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
plt.title('Forecast Error Comparison: Large vs Small Firms (All Models)')
plt.xlabel('Model')
plt.ylabel('Median Absolute Forecast Error')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

What the grouped bar chart shows is that the absolute forecast errors are higher for the Large group than for the Small group—and this holds across the board, from the random walk models (RW_1 through RW_3) to the moving average models (MA_2 through MA_5). But the real story here isn't that the models do a worse job predicting for large firms. It's simply a scale effect baked into the absolute error metric itself.

Absolute error is calculated as `eps_t1 - eps`, and its unit is the same as EPS itself—dollars per share. Large firms tend to have much higher EPS in absolute terms.A large firm might have EPS of $5, while a small firm might have EPS of $0.50. Even if both firms have the same 10% forecast error in relative terms, the large firm ends up with an absolute error of $0.50 ($5 × 10%), while the small firm clocks in at just $0.05 ($0.50 × 10%). The large firm's error will always appear bigger. So raw absolute error, without any standardization, is not a fair for comparing forecasting accuracy across firms of different sizes.

#### Scaled Forecast Error Analysis

To get around the scale problem with absolute errors, we need an error metric that doesn't depend on firm size—something that lets us compare forecasting accuracy fairly across large and small firms. So we bring in the scaled (standardized) error, calculated as:

`scaled_error = abs(eps_t1 - eps) / abs(eps)`

What this formula does is take the absolute forecast error and divide it by the absolute value of current-period EPS. The result is the error expressed as a fraction of current earnings—a relative error. 

On the implementation side, we compute the `scaled_error` for each of the three random walk models and three moving average models. To handle division-by-zero issues where EPS is near or exactly zero, we use `replace([inf, -inf], np.nan)` to swap out any infinities for missing values. After that, we clean up the data with `dropna`, group by `size_group`, and compute the median scaled error for each group. We then visualize the results using the same melt + barplot approach as before.

In [ ]:
df['scaled_error_rw_1'] = abs(df['eps_t1'] - df['eps']) / abs(df['eps'])
df['scaled_error_rw_2'] = abs(df['eps_t2'] - df['eps']) / abs(df['eps'])
df['scaled_error_rw_3'] = abs(df['eps_t3'] - df['eps']) / abs(df['eps'])

df['scaled_error_ma_2'] = abs(df['eps_t1'] - df['ma_2']) / abs(df['eps'])
df['scaled_error_ma_3'] = abs(df['eps_t1'] - df['ma_3']) / abs(df['eps'])
df['scaled_error_ma_5'] = abs(df['eps_t1'] - df['ma_5']) / abs(df['eps'])

scaled_cols = ['scaled_error_rw_1', 'scaled_error_rw_2', 'scaled_error_rw_3', 
               'scaled_error_ma_2', 'scaled_error_ma_3', 'scaled_error_ma_5']
df[scaled_cols] = df[scaled_cols].replace([np.inf, -np.inf], np.nan)

sub_df_scaled = df.dropna(subset=scaled_cols + ['size_group'])

median_scaled_errors = sub_df_scaled.groupby('size_group')[scaled_cols].median()

display(median_scaled_errors)

median_scaled_errors_reset = median_scaled_errors.reset_index().melt(
    id_vars='size_group', var_name='Model', value_name='Scaled Absolute Error'
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=median_scaled_errors_reset, x='Model', y='Scaled Absolute Error',
            hue='size_group', palette='Set2')
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f")
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
plt.title('Forecast Error Comparison: Large vs Small Firms (Scaled/Standardized)')
plt.xlabel('Model')
plt.ylabel('Median Scaled Absolute Forecast Error')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

The bar chart for scaled errors paints a completely different picture from the absolute errors. Across all six forecasting models, the Large group shows noticeably lower relative forecast errors than the Small group. What this reveals is that, once you standardize for scale, large-company earnings are actually more predictable. We analyzed three possible causes of this problem:

First, earnings persistence. Large firms tend to be mature businesses with stable market shares, diversified revenue streams, and well-established operating models. Their core profitability holds fairly steady from year to year. Current EPS simply has more explanatory power for future EPS, so history-based forecasting models naturally perform better on large firms.

Second, differences in the information environment. Large firms generally attract more analyst coverage and market scrutiny, face stricter disclosure requirements, and have less room for noise or accrual manipulation in their reported earnings. Higher-quality earnings information means current-period data reflects the underlying fundamentals more faithfully, which in turn helps improve forecast accuracy.

Third, the higher volatility of small firms. Compared to large firms, small firms are far more vulnerable to external shocks—losing a single customer, shifts in a product lifecycle, financing constraints—all of which can cause EPS to swing wildly from year to year. Even if the absolute dollar change in EPS is small, the proportional swing relative to a tiny EPS base is often large, and that directly drives up their scaled forecast errors.

To sum it up, the standardized error analysis shows that, on a relative basis, forecasting models actually produce smaller errors for large firms. That's driven by their stronger earnings persistence, better information environment, and more stable business characteristics.

## Part 4: Complex Models

### OLS Regression

We'll use the following model for the regression:

$$
ROA_{t+1} = \beta_0 + \beta_1 ROA_t + \beta_2 PM_t + \beta_3 Loss_t + \beta_4 \log AT_t + \beta_5 ATO_t + \varepsilon
$$

wip-how-and-why-these-code (0862)

In [ ]:
import statsmodels.formula.api as smf

formula_hybrid = "roa_t1 ~ roa_t + pm_t + loss_t + log_at + ato_t"
model_hybrid = smf.ols(formula_hybrid, data=df).fit()
print("  Hybrid Model: roa_t1 ~ roa_t + pm_t + loss_t + log_at + ato_t")
model_hybrid.summary()

The OLS regression results provide empirical evidence on the factors that drive future profitability. The model is jointly significant (F-statistic = 9047, p ≈ 0.00), and the five explanatory variables together explain roughly 42.4% of the variation in future ROA (R² = 0.424, Adj. R² = 0.424). With 61,511 observations, R² and Adj. R² are virtually identical, so overfitting is not a concern. Here is how to interpret each coefficient.

The coefficient on roa_t is 0.3967 (t = 95.411), by far the strongest predictor in the model. If a firm's current ROA is one percentage point above the mean, its expected future ROA is only about 0.40 percentage points above the mean — the remaining 60% reverts within a single year. This decay rate carries important economic content: competitive advantage is transient. High profitability invites competitor entry, strengthens supplier bargaining power, attracts regulatory scrutiny, and prompts customers to seek alternatives. These market forces collectively erode excess returns. Conversely, loss-making firms do not necessarily remain unprofitable — exit, restructuring, management turnover, and asset divestiture push them back toward the mean as well. Among all observable financial information, current profitability level concentrates almost every predictable signal about the future. The other variables add statistically significant but economically modest incremental information.

The coefficients on pm_t and ato_t capture the two components of the DuPont decomposition. ROA is, by construction, the product of profit margin and asset turnover: ROA = PM × ATO. This regression simultaneously includes roa_t (the product) and both pm_t and ato_t (the two components), so the specification essentially asks: once we know the overall return on assets, does decomposing it into margin and turnover components provide additional predictive power? The answer is yes, but the increment is modest — pm_t has a coefficient of 0.0234 (t = 34.598) and ato_t has 0.0236 (t = 33.931). The two coefficients are nearly identical, which is itself an economically meaningful finding. After controlling for the overall ROA level, profit margin and asset turnover contribute symmetrically to predicting future profitability — the market does not favor "high-margin, low-turnover" differentiation strategies over "low-margin, high-turnover" efficiency strategies. The two paths to ROA are equally weighted in their predictive content. More importantly, the combined magnitude of the two component coefficients (roughly 0.047) is far smaller than the coefficient on roa_t (0.397), indicating that ROA's predictive power comes primarily from the joint effect of the two components — their product — rather than from their separate linear contributions. This makes sense economically: margin and turnover involve a trade-off, since firms rarely achieve both simultaneously. Optimizing along one dimension often sacrifices the other, and ROA, as their product, automatically internalizes these trade-offs.

The coefficient on loss_t is -0.0385 (t = -26.217), and this is after controlling for roa_t. The logic deserves careful unpacking. roa_t is a continuous variable that already captures the degree of loss — losing $1 versus losing $100 shows up in different roa_t values. So why does loss_t, a simple 0/1 binary indicator, provide additional explanatory power beyond the continuous measure? This points to a discontinuity near zero profit: crossing from slightly positive to slightly negative is not a marginal change in degree, but a qualitative jump in state. The reason lies in the asymmetric treatment of the profit/loss label by market institutions and contractual arrangements. Bank financial covenants typically trigger upon negative net income; executive compensation plans often use positive earnings as a performance threshold; auditors apply heightened going-concern scrutiny to loss firms; suppliers may tighten trade credit; and the capital market's negative reaction to the loss label may itself worsen operating conditions. roa_t captures the magnitude of the loss; loss_t captures the additional penalty imposed by the loss label itself. The marginal effect of -0.0385 means that, holding all else equal, merely carrying the label "loss firm" reduces expected future ROA by an additional 0.039 units.

The coefficient on log_at is 0.0052 (t = 20.983) — highly significant statistically, but modest economically. Doubling total assets (increasing log_at by approximately 0.693) raises expected future ROA by less than 0.004 units. Size per se is not a core driver of profitability. The theoretical advantages of large firms — economies of scale, market power, diversification, lower financing costs, deeper talent pools — translate into very little incremental future profitability once current profitability and operating efficiency (roa_t, pm_t, ato_t) are controlled for. This aligns with a core proposition of industrial organization economics: if size alone delivered sustained excess returns, every industry would tend toward monopoly; in reality, we observe oligopolistic competition and the coexistence of many mid-sized firms. That said, the fact that the log_at coefficient is positive and significant, however small, does carry its own implication: conditional on observable variables being equal, large firms are somewhat safer and more predictable than small ones. Investors demand higher risk premia when pricing small firms, and analyst forecast dispersion is wider for small firms — both consistent with the regression finding that large firms exhibit systematically, albeit modestly, higher future ROA.

The intercept is -0.0278 (t = -13.974), describing a hypothetical firm with ROA = 0, profit margin = 0, non-loss, infinitesimal total assets, and zero asset turnover. For a firm with virtually no positive operating signals, an expected negative future ROA is reasonable. The more noteworthy point is the intercept's significance: when all explanatory variables are at zero, the market's rational expectation is not break-even but unambiguously negative. This echoes the accounting principle of conservatism — absent any positive evidence, the baseline assumption about a firm's prospects should be pessimistic.

The R² of 0.424 sits at an economically plausible level. If R² were far above 0.8, it would suggest financial ratios are nearly mechanically deterministic, with genuine economic forces like competition, technology shocks, and managerial decisions reduced to irrelevance. If it were below 0.1, it would suggest financial information is essentially useless. At 0.424, historical financial statements contain substantial information about the future, but the remaining 57.6% represents genuine unpredictability — the success or failure of new products, competitors' preemptive moves, macroeconomic policy shifts, supply chain disruptions, and unexpected management departures. This captures the learning rate of uncertainty in a market economy: roughly 42% of next year's profitability is already written into today's financial statements, while 58% is left to tomorrow.

On the diagnostics side, the Durbin-Watson statistic of 1.942, close to 2, indicates no significant residual autocorrelation, providing a reliable foundation for statistical inference on coefficient significance. The condition number of 54.7 falls within an acceptable range — although roa_t, pm_t, and ato_t are mathematically related through a product relationship, collinearity is not severe enough to threaten estimate stability because they enter the regression in additive form. The residual kurtosis of 14.455 (versus 3 for a normal distribution) reveals heavy tails — the sample contains firms whose prediction errors deviate substantially from the fitted line. For a large sample of 61,511 observations, fat tails are a routine feature of financial data and do not affect the consistency or unbiasedness of the OLS coefficients. They do, however, warrant caution when interpreting means versus medians in any extrapolation.


### Exploring Data

#### Time-Series Forecast Models

The code below constructs the absolute forecast error variable `error_rw` for the random walk model, which serves as the baseline time-series benchmark against which all other forecasting models are compared.

Computation logic:

The random walk model assumes that, absent any other information, next period's earnings approximate current period's earnings: $NI_{t+1} \approx NI_t$. The forecast error is therefore defined as the absolute deviation of actual $NI_{t+1}$ from the random walk prediction $NI_t$:

$$error\_rw = \frac{|NI_{t+1} - NI_t|}{AT_t}$$

The numerator `abs(df['ni1'] - df['ni'])` computes the absolute difference between next year's net income (`ni1`, constructed earlier via `groupby('gvkey')['ni'].shift(-1)`) and the current year's net income (`ni`). This captures the raw dollar forecasting error produced by the random walk model.

The denominator `df['at']` — current total assets — serves a critical scaling function. Without it, the forecast error would be denominated in absolute dollars, making it inherently larger for big firms than for small firms regardless of actual forecasting accuracy. This is the same scale effect discussed later in the Scaled Forecast Error Analysis section: a $500 million error at a mega-cap firm may represent a 2% miss, while a $10 million error at a small-cap firm could represent a 30% miss, yet the absolute figure would misleadingly suggest the large firm's forecast was worse. Dividing by total assets converts the error into an ROA-like ratio, neutralizing the scale distortion and enabling meaningful comparison across firms of different sizes.

Why total assets rather than EPS or another scaling variable? This choice aligns with the broader modeling framework of the project. The core OLS regression model's dependent variable $ROA_{t+1}$ and its key regressor $ROA_t$ are both standardized by total assets. Using the same denominator for the random walk error preserves dimensional consistency. When the two models are later compared side by side — in the `error_full` table under the Regression Analysis section where Median AE and Mean AE of the Hybrid Model and Random Walk are juxtaposed — both error metrics share a common unit, making the comparison straightforward and scientifically valid.

Subsequent use: The variable `error_rw` is later referenced in the Model Comparison Visualization section, where it is compared against the Hybrid OLS model's prediction error `error_hybrid` — both for the full sample and in the pre-2010 / post-2010 sub-period analysis — to evaluate whether incorporating additional financial variables improves forecasting accuracy over the random walk baseline.

In [ ]:
df['error_rw'] = abs(df['ni1'] - df['ni']) / df['at']
df.head()

#### Comparing the forecast errors

wip-how-and-why-these-code (1142)

In [ ]:
df["pred_hybrid"]  = model_hybrid.predict(df)

df["error_hybrid"]  = abs(df["roa_t1"] - df["pred_hybrid"])

error_full = pd.DataFrame({
    "Model":     ["Hybrid", "Random Walk"],
    "Median AE": [df["error_hybrid"].median(),
                  df["error_rw"].median()],
    "Mean AE":   [df["error_hybrid"].mean(),
                  df["error_rw"].mean()]
})
display(error_full)

pre_2010  = df[df["fyear"] < 2010]
post_2010 = df[df["fyear"] >= 2010]

print(f"Pre-2010 observations:  {len(pre_2010):,}")
print(f"Post-2010 observations: {len(post_2010):,}")

period_errors = pd.DataFrame({
    "Model":     ["Hybrid", "Random Walk",
                  "Hybrid", "Random Walk"],
    "Period":    ["Pre-2010", "Pre-2010",
                  "Post-2010", "Post-2010"],
    "Median AE": [
        pre_2010["error_hybrid"].median(),
        pre_2010["error_rw"].median(),
        post_2010["error_hybrid"].median(),
        post_2010["error_rw"].median()
    ]
})
display(period_errors)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=error_full, x="Model", y="Median AE",
            ax=axes[0], palette="Set2", hue="Model", legend=False)
for c in axes[0].containers:
    axes[0].bar_label(c, fmt="%.4f")
axes[0].set_ylim(0, axes[0].get_ylim()[1] * 1.15)
axes[0].set_title("Median Absolute Error (Full Sample)")
axes[0].set_ylabel("Median AE")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=period_errors, x="Model", y="Median AE",
            hue="Period", ax=axes[1], palette="Set2")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.4f")
axes[1].set_ylim(0, axes[1].get_ylim()[1] * 1.15)
axes[1].set_title("Median Absolute Error by Period")
axes[1].set_ylabel("Median AE")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

wip-description-of-the-result (1212)

result: 
Model	Period	Median AE
0	Hybrid	Pre-2010	0.043157
1	Random Walk	Pre-2010	0.037447
2	Hybrid	Post-2010	0.029598
3	Random Walk	Post-2010	0.021607


Model	Median AE	Mean AE
0	Hybrid	0.036665	0.073838
1	Random Walk	0.030149	0.095817


## Summary

wip-description-summary (1242)

## AI Statement

AI tools were used to assist with code structure, debugging, and markdown explanations. All outputs were verified for accuracy, economic reasoning, and compliance with project requirements.